In [113]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import seaborn as sns
import jupyterlab as jl
import re

df = pd.read_csv('Motor_Vehicle_Collisions_-_Crashes.csv', low_memory=False )

print(df.shape)
df.info()


(449586, 29)
<class 'pandas.DataFrame'>
RangeIndex: 449586 entries, 0 to 449585
Data columns (total 29 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   CRASH DATE                     449586 non-null  str    
 1   CRASH TIME                     449586 non-null  str    
 2   BOROUGH                        348242 non-null  str    
 3   ZIP CODE                       322555 non-null  float64
 4   LATITUDE                       425039 non-null  float64
 5   LONGITUDE                      425039 non-null  float64
 6   LOCATION                       425039 non-null  str    
 7   ON STREET NAME                 319059 non-null  str    
 8   CROSS STREET NAME              130523 non-null  str    
 9   OFF STREET NAME                232292 non-null  str    
 10  NUMBER OF PERSONS INJURED      449586 non-null  int64  
 11  NUMBER OF PERSONS KILLED       449586 non-null  int64  
 12  NUMBER OF PEDESTRIANS INJURE

# Data Issues and Cleaning Log

**Dataset:** NYC Motor Vehicle Collisions - Crashes (NYC Open Data)
**Size as downloaded:** 449,586 rows, 29 columns. Each row is one crash.

**1. Crash date and time are stored as text**
- **Decision:** Convert to datetime and combine into one column. Check the result with `min()` and `max()`. Extract hour, month and year into their own columns.
- **Why:** Needed for filtering, sorting and the time analysis (Q1, Q6).

In [114]:
df_raw = df.copy()  # Keep a copy of the raw data for reference

df['CRASH INFO'] = pd.to_datetime(df['CRASH DATE'] + ' ' + df['CRASH TIME'], format="%m/%d/%Y %H:%M", errors='coerce')  # Combine date and time into a single datetime column

df['CRASH YEAR'] = df['CRASH INFO'].dt.year
df['CRASH MONTH'] = df['CRASH INFO'].dt.month
df['CRASH HOUR'] = df['CRASH INFO'].dt.hour

df['CRASH INFO'].isna().sum()  # Check for any rows where the datetime conversion failed

np.int64(0)

**2. ZIP code is stored as a decimal number**
- **Decision:** Convert to text.
- **Why:** A ZIP code is an identifier, not a quantity.

**7. ZIP codes may be 0 or outside New York City**
- **Decision:** Check for 0 and for values outside the NYC range. Set invalid values to missing.
- **Why:** 0 is a placeholder, not a real ZIP.

In [115]:
df["ZIP CODE"] = df["ZIP CODE"].astype("Int64").astype("string")
df["ZIP CODE"].str.len().value_counts()

zip_num = pd.to_numeric(df["ZIP CODE"], errors="coerce") 

in_range = zip_num.between(10000, 11698)
missing  = zip_num.isna()

outside_range = ~in_range & ~missing

(df['ZIP CODE'] == "0").sum()  # Count how many ZIP codes are "0"

df.loc[outside_range, 'ZIP CODE'] = pd.NA  # Set ZIP codes outside the valid range to missing






**3. Collision ID is stored as text**
- **Decision:** Check that it is unique per crash.
- **Why:** Duplicate crashes would inflate every count.

**4. Possible duplicate collision IDs**
- **Decision:** Remove only if the rows are identical. If the same ID has different data, investigate first.
- **Why:** Avoid deleting real records.

In [116]:
df["COLLISION_ID"].is_unique
df.duplicated(subset=["COLLISION_ID"]).sum()  # Count how many duplicate COLLISION_IDs there are

np.int64(0)

## 2. Duplicates and consistency

**5. Injury and death totals may not match their parts**
- **Decision:** The total of persons injured exceeds the sum of pedestrians, cyclists and motorists in 12,404 rows (120 rows for killed), and never falls below it. The difference is kept as a fourth group, "other" (`OTHER INJURED`, `OTHER KILLED`), so the four groups add up to the total.
- **Why:** The totals are used in Q2 and the groups in Q4. Keeping the gap as its own group avoids dropping people from the analysis.

In [117]:
# Decision: the gap between the total and the three named groups is kept as a fourth group, "other"
for kind in ["INJURED", "KILLED"]:
    parts = [f"NUMBER OF {g} {kind}" for g in ["PEDESTRIANS", "CYCLIST", "MOTORIST"]]
    total = f"NUMBER OF PERSONS {kind}"
    df[f"OTHER {kind}"] = df[total] - df[parts].sum(axis=1)
    print(kind,
          "| rows with an 'other' group:", (df[f"OTHER {kind}"] > 0).sum(),
          "| people in 'other':", df[f"OTHER {kind}"].sum(),
          "| negative gaps:", (df[f"OTHER {kind}"] < 0).sum())

INJURED | rows with an 'other' group: 12404 | people in 'other': 13032 | negative gaps: 0
KILLED | rows with an 'other' group: 120 | people in 'other': 121 | negative gaps: 0


**6. Borough is missing in many rows (about 22.5%)**
- **Decision:** Decide after checking. Options: label as "Unknown" for borough analyses, recover from latitude and longitude, or exclude from those analyses only.
- **Why:** Q1, Q2 and Q5 depend on borough. These crashes still count for the other questions.

In [119]:
df["BOROUGH"] = df["BOROUGH"].fillna("Unknown")
print(df["BOROUGH"].value_counts())

BOROUGH
BROOKLYN         118822
Unknown          101344
QUEENS            95258
MANHATTAN         63234
BRONX             56997
STATEN ISLAND     13931
Name: count, dtype: int64


**8. Latitude and longitude may be 0 or outside New York City**
- **Decision:** Check for 0 and for values outside the NYC range. Set invalid values to missing.
- **Why:** They would plot far outside the city.

In [121]:
NYC_LAT = (40.49, 40.92)
NYC_LON = (-74.27, -73.68) 

valid = df["LATITUDE"].between(*NYC_LAT) & df["LONGITUDE"].between(*NYC_LON)
has_coords = df["LATITUDE"].notna() & df["LONGITUDE"].notna()

bad = ~valid & has_coords
df.loc[bad, ["LATITUDE", "LONGITUDE"]] = pd.NA # Set invalid coordinates to missing

df["LATITUDE"].isna().sum(), df["LONGITUDE"].isna().sum()  # Check how many missing coordinates there are now

(np.int64(30885), np.int64(30885))

**9. Some street names are only numbers, and some may be placeholders such as 0**
- **Decision:** Check with `value_counts()`. Low priority.
- **Why:** None of the 6 questions use street names. Mention it in the limitations.

In [122]:
street_cols = ["ON STREET NAME", "CROSS STREET NAME", "OFF STREET NAME"]
for col in street_cols:
    df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)
    n_placeholder = (df[col] == "0").sum()
    df.loc[df[col] == "0", col] = pd.NA
    print(f"{col}: {n_placeholder} placeholder '0' values set to missing")

ON STREET NAME: 0 placeholder '0' values set to missing
CROSS STREET NAME: 0 placeholder '0' values set to missing
OFF STREET NAME: 3 placeholder '0' values set to missing


**10. Contributing factors include "Unspecified" and "Does Not Apply"**
- **Decision:** Keep both categories as they appear in the data. State their share when reporting Q3.
- **Why:** They are recorded values and removing them would drop many crashes from the ranking. They are not actual causes, so their share needs to be visible when reading the results.

In [123]:
contributing_factor_cols = [f"CONTRIBUTING FACTOR VEHICLE {i}" for i in range(1, 6)]

for col in contributing_factor_cols:
    print(df[col].value_counts(dropna=False).head(10), "\n")

CONTRIBUTING FACTOR VEHICLE 1
Driver Inattention/Distraction    93333
Does Not Apply                    82881
Unspecified                       76458
Failure to Yield Right-of-Way     27638
Following Too Closely             21701
Passing or Lane Usage Improper    15403
Other Vehicular                   14213
Unsafe Speed                      13430
Passing Too Closely               12458
Backing Unsafely                  11180
Name: count, dtype: int64 

CONTRIBUTING FACTOR VEHICLE 2
Unspecified                       202916
NaN                               104028
Does Not Apply                     43234
Driver Inattention/Distraction     36082
Following Too Closely               9910
Other Vehicular                     6557
Passing or Lane Usage Improper      6428
Failure to Yield Right-of-Way       5566
Passing Too Closely                 4753
Unsafe Speed                        4317
Name: count, dtype: int64 

CONTRIBUTING FACTOR VEHICLE 3
NaN                               409813
Uns

**11. Vehicle type labels differ by case and spelling**
- **Example:** sedan, SEDAN, sedan 2, sedan 4.
- **Decision:** Lowercase, trim spaces, and group into a short list of categories. Keep the raw column and document the grouping rules.
- **Why:** Needed for the vehicle ranking in Q5.

In [124]:
rules = [
    ("unknown",              r"^(unknown|unk)( (car|truck|vehicle))?$"),
    ("e-bike / e-scooter",   r"e-?bike|e-?scooter|standing scooter|electric"),
    ("bicycle",              r"^bike$|bicycl"),
    ("motorcycle / moped",   r"motorcycle|moped|motorscooter|motorbike|dirt bike|^scooter"),
    ("emergency",            r"ambul|fire|police"),
    ("bus",                  r"bus"),
    ("taxi / limo",          r"taxi|limo|livery"),
    ("pickup",               r"pick-?up|^pk$"),
    ("van",                  r"van"),
    ("truck",                r"truck|tractor|box|dump|flat ?bed|flat rack|garbage|refuse|tanker|concrete|chassis|delivery|trailer|tow|plow|armored|carry all"),
    ("car / SUV",            r"sedan|station wagon|sport utility|suv|convertible|subn|3-door|2-door|coupe|wagon|utility"),
]

def categorize(label):
    if pd.isna(label):
        return pd.NA
    for name, pattern in rules:
        if re.search(pattern, label):
            return name
    return "other"


vehicle_cols = [f"VEHICLE TYPE CODE {i}" for i in range(1, 6)]
for i, col in enumerate(vehicle_cols, start=1):
    clean = (df[col].str.strip().str.lower().str.replace(r"\s+", " ", regex=True))
    mapping = {v: categorize(v) for v in clean.dropna().unique()}   # only ~1,200 labels
    df[f"VEHICLE CLEAN {i}"] = clean
    df[f"VEHICLE CAT {i}"] = clean.map(mapping)

**12. Other text columns may have inconsistent case or extra spaces**
- **Decision:** Apply the same case and spacing checks to borough and contributing factors.
- **Why:** Prevents one category from appearing under several names.

In [125]:
text_cols = ["BOROUGH"] + [f"CONTRIBUTING FACTOR VEHICLE {i}" for i in range(1, 6)]
for col in text_cols:
    before = df[col].nunique()
    df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)
    print(f"{col}: {before} -> {df[col].nunique()} unique values")

BOROUGH: 6 -> 6 unique values
CONTRIBUTING FACTOR VEHICLE 1: 64 -> 64 unique values
CONTRIBUTING FACTOR VEHICLE 2: 64 -> 64 unique values
CONTRIBUTING FACTOR VEHICLE 3: 52 -> 52 unique values
CONTRIBUTING FACTOR VEHICLE 4: 38 -> 38 unique values
CONTRIBUTING FACTOR VEHICLE 5: 30 -> 30 unique values


**13. Location repeats latitude and longitude**
- **Decision:** Confirm it is missing in exactly the same rows as latitude and longitude, then drop it.
- **Why:** It is redundant.

In [126]:
loc = (df["LOCATION"].str.strip()
         .str.extract(r"^\(\s*(-?[\d.]+)\s*,\s*(-?[\d.]+)\s*\)$")
         .astype(float))
loc.columns = ["loc_lat", "loc_lon"]

# 2. Compare to the real columns, only where LOCATION exists
has_loc = df["LOCATION"].notna()
lat_match = (loc["loc_lat"] - df["LATITUDE"]).abs() < 1e-9
lon_match = (loc["loc_lon"] - df["LONGITUDE"]).abs() < 1e-9

print("rows with LOCATION:", has_loc.sum())
print("latitude matches:", lat_match[has_loc].sum())
print("longitude matches:", lon_match[has_loc].sum())
print("both match:", (lat_match & lon_match)[has_loc].sum())
print("mismatches:", (~(lat_match & lon_match) & has_loc).sum())

# 3. Missing together?
print("coords present but LOCATION missing:", (df["LATITUDE"].notna() & ~has_loc).sum())
print("LOCATION present but coords missing:", (has_loc & df["LATITUDE"].isna()).sum())

df = df.drop(columns=["LOCATION"])  # Drop the LOCATION column since we have separate LATITUDE and LONGITUDE columns

rows with LOCATION: 425039
latitude matches: 418701
longitude matches: 418701
both match: 418701
mismatches: 6338
coords present but LOCATION missing: 0
LOCATION present but coords missing: 6338



**14. No column marks crashes with injuries**
- **Decision:** Create `injury_crash` (True if at least one person was injured).
- **Why:** Needed for Q3.

In [127]:
df["INJURY CRASH"] = df["NUMBER OF PERSONS INJURED"] > 0

print("Number of injury crashes:", df["INJURY CRASH"].sum())

Number of injury crashes: 188800


## Save the cleaned data
- **Decision:** Write the cleaned table to a new file in `data/clean/`. The raw CSV is never overwritten.
- **Why:** The analysis section can start from the cleaned file without repeating the cleaning. CSV files are ignored by `.gitignore`, so this file is not pushed to GitHub.

In [128]:
from pathlib import Path

Path("data/clean").mkdir(parents=True, exist_ok=True)

print("Rows before cleaning:", df_raw.shape[0], "| after:", df.shape[0], "| rows removed:", df_raw.shape[0] - df.shape[0])
print("Columns before cleaning:", df_raw.shape[1], "| after:", df.shape[1])

df.to_csv("data/clean/collisions_clean.csv", index=False)
print("Saved: data/clean/collisions_clean.csv")

# When reloading, keep ZIP CODE as text so it does not become a number again:
# pd.read_csv("data/clean/collisions_clean.csv", dtype={"ZIP CODE": "string"}, low_memory=False)

Rows before cleaning: 449586 | after: 449586 | rows removed: 0
Columns before cleaning: 29 | after: 45
Saved: data/clean/collisions_clean.csv


In [129]:

df = pd.read_csv('data/clean/collisions_clean.csv', low_memory=False )

print(df.shape)
df.info()

(449586, 45)
<class 'pandas.DataFrame'>
RangeIndex: 449586 entries, 0 to 449585
Data columns (total 45 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   CRASH DATE                     449586 non-null  str    
 1   CRASH TIME                     449586 non-null  str    
 2   BOROUGH                        449586 non-null  str    
 3   ZIP CODE                       322554 non-null  float64
 4   LATITUDE                       418701 non-null  float64
 5   LONGITUDE                      418701 non-null  float64
 6   ON STREET NAME                 319059 non-null  str    
 7   CROSS STREET NAME              130523 non-null  str    
 8   OFF STREET NAME                232289 non-null  str    
 9   NUMBER OF PERSONS INJURED      449586 non-null  int64  
 10  NUMBER OF PERSONS KILLED       449586 non-null  int64  
 11  NUMBER OF PEDESTRIANS INJURED  449586 non-null  int64  
 12  NUMBER OF PEDESTRIANS KILLED